In [ ]:
# Instalación y login GEE
!pip install earthengine-api -q

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='earthengine-cafe')

In [ ]:
# 2) Parámetros básicos
start_date = '2023-01-01'
end_date   = '2025-11-01'

# Admin-1 para Cauca y Narino desde GAUL
gaul = ee.FeatureCollection("FAO/GAUL/2015/level1")

cauca  = gaul.filter(ee.Filter.And(ee.Filter.eq('ADM0_NAME', 'Colombia'), ee.Filter.eq('ADM1_NAME', 'Cauca'))).geometry()
narino = gaul.filter(ee.Filter.And(ee.Filter.eq('ADM0_NAME', 'Colombia'), ee.Filter.eq('ADM1_NAME', 'Narino'))).geometry()

regions = [
    {'name': 'Cauca',  'geom': cauca},
    {'name': 'Narino', 'geom': narino}
]

# Colecciones de datos
# ============================
# ERA5-Land diario
era5 = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR").filterDate(start_date, end_date))

# MODIS NDVI/EVI (16 días)
modis_vi = (ee.ImageCollection("MODIS/061/MOD13Q1").filterDate(start_date, end_date).select(['NDVI', 'EVI']))

# FLDAS ET mensual (Evap_tavg)
fldas = (ee.ImageCollection("NASA/FLDAS/NOAH01/C/GL/M/V001").filterDate(start_date, end_date))


# Transformaciones de datos
# ============================
# Añadir NDVI/EVI "diarios" (toma el MODIS más cercano ±8 días)
def add_vi(image):
    date = image.date()
    vi_img = (
        modis_vi
        .filterDate(date.advance(-8, 'day'), date.advance(8, 'day'))
        .sort('system:time_start')
        .first()
    )
    # Si no hay MODIS cercano, pone 0
    vi_img = ee.Image(
        ee.Algorithms.If(
            vi_img,
            vi_img,
            ee.Image.constant([0, 0]).rename(['NDVI', 'EVI']).toFloat()
        )
    )
    return image.addBands(vi_img)

era5_with_vi = era5.map(add_vi)


# Preparar bandas derivadas ERA5 + MODIS
# ============================
def prep_bands(image):
    # Temperaturas K -> °C
    tmax_c   = image.select('temperature_2m_max').subtract(273.15).rename('tmax_c')
    tmin_c   = image.select('temperature_2m_min').subtract(273.15).rename('tmin_c')
    tmean_c  = image.select('temperature_2m').subtract(273.15).rename('tmean_c')
    dew_c    = image.select('dewpoint_temperature_2m').subtract(273.15).rename('dewpoint_c')

    # Precipitación y PET: m -> mm
    precip_mm = image.select('total_precipitation_sum').multiply(1000).rename('precip_mm')
    pet_mm    = image.select('potential_evaporation_sum').multiply(1000).rename('pet_mm')

    # Humedad del suelo (0–1)
    soil_moist = image.select('volumetric_soil_water_layer_1').rename('soil_moist_layer1')

    # Índice de vegetación tipo LAI
    lai_high = image.select('leaf_area_index_high_vegetation').rename('lai_high')

    # NDVI/EVI MODIS: escalar por 0.0001
    ndvi = image.select('NDVI').multiply(0.0001).rename('ndvi')
    evi  = image.select('EVI').multiply(0.0001).rename('evi')

    out = (precip_mm
           .addBands([tmax_c, tmin_c, tmean_c, dew_c,
                      pet_mm, soil_moist, lai_high,
                      ndvi, evi])
           .copyProperties(image, ['system:time_start'])
          )
    return out

era5_prepared = era5_with_vi.map(prep_bands)

#  Añadir ET real de FLDAS (mm/día)
#   Se usa la ET media del mes, constante para todos los días del mes:
#   et_mm = Evap_tavg (kg/m2/s) * 86400  -> mm/día
def add_et(image):
    date = image.date()
    month_range = date.getRange('month')
    month_start = month_range.start()
    month_end   = month_range.end()

    et_img = (fldas.filterDate(month_start, month_end).select('Evap_tavg').first())

    et_mm = ee.Image(
        ee.Algorithms.If(
            et_img,
            ee.Image(et_img).multiply(86400).rename('et_mm'),
            ee.Image.constant(0).rename('et_mm')
        )
    )

    return image.addBands(et_mm)

era5_with_et = era5_prepared.map(add_et)

band_names = [
    'precip_mm',
    'tmax_c',
    'tmin_c',
    'tmean_c',
    'dewpoint_c',
    'pet_mm',
    'soil_moist_layer1',
    'lai_high',
    'ndvi',
    'evi',
    'et_mm'
]

# Reducir por región y día
def extract_for_region(region_dict):
    geom = region_dict['geom']
    name = region_dict['name']

    def per_image(image):
        stats = image.select(band_names).reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geom,
            scale=10000,      # ~10 km, compatible con ERA5-Land y FLDAS
            maxPixels=1e13
        )
        return (
            ee.Feature(None, stats)
            .set('date', image.date().format('YYYY-MM-dd'))
            .set('region', name)
        )

    return era5_with_et.map(per_image)

cauca_fc  = extract_for_region(regions[0])
narino_fc = extract_for_region(regions[1])

table = cauca_fc.merge(narino_fc)


# Exportar a Google Drive
task = ee.batch.Export.table.toDrive(
    collection = table,
    description = 'Cauca_Narino_ERA5L_MODIS_FLDAS_daily_2023_2025',
    fileFormat = 'CSV'
)

task.start()


Export task started. Revisa tu Google Drive cuando termine.
